# NITSCI v2: High-Accuracy DR Grading on IDRiD

Upgrades over v1, all aimed at accuracy. Every number is produced by real training and a held-out test set:

| # | Technique | Why it helps |
|---|-----------|--------------|
| 1 | **Transfer learning from APTOS-2019** (3,662 labelled fundus images, same 0–4 scale) | IDRiD alone (455 images) is too small; pretraining on 8× more retinal data is the single largest gain |
| 2 | **EfficientNetV2-S (ImageNet-21k)** backbone at **448 px** | stronger features, higher resolution for microaneurysms |
| 3 | **Ben Graham preprocessing** (local-contrast normalisation, used by APTOS top solutions) | removes lighting differences between cameras |
| 4 | **Dual head: classification + ordinal regression** | DR grades are ordered; regression reduces "off-by-one" errors |
| 5 | **Optimised decision thresholds** (fitted on out-of-fold predictions) | directly maximises accuracy on unseen folds |
| 6 | **EMA weights** | smoother, better-generalising final weights |
| 7 | **5-fold cross-validation ensemble + test-time augmentation (4 flips)** | averages 20 predictions per image |
| 8 | Cross-expert lesion-attention architecture from v1 (global expert, lesion expert, cross-attention, adaptive gate) | kept as the core model |

**Two tasks are reported:**
- **5-class DR grading** (0–4): the hardest task.
- **Referable DR** (grade ≥ 2 vs < 2): the clinical screening task. >90% accuracy is realistic here.

**Runtime:** use a **T4 GPU** (*Runtime → Change runtime type*). A full run takes about 1–1.5 h. Set `FAST = True` for a ~20-minute check run.

## 1. Setup and configuration

In [ ]:
!pip install -q timm grad-cam kagglehub

import os, re, glob, random, copy, time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import timm
import kagglehub
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, cohen_kappa_score

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'NO GPU')
AMP = device.type == 'cuda'

FAST = False                       # True = quick check run (fewer epochs/folds)

BACKBONE      = 'tf_efficientnetv2_s.in21k_ft_in1k'
IMG_SIZE      = 448
PREPROC       = 'ben'              # 'ben' (Ben Graham) or 'clahe' (CLAHE + bilateral, as in v1)
BATCH_SIZE    = 16
NUM_CLASSES   = 5
USE_APTOS     = True               # pretrain on APTOS-2019 before IDRiD
APTOS_EPOCHS  = 3 if FAST else 10
IDRID_EPOCHS  = 8 if FAST else 25
N_FOLDS       = 2 if FAST else 5
PATIENCE      = 8
REG_WEIGHT    = 0.5                # weight of the ordinal-regression loss
SPARSITY      = 0.01               # keeps lesion-attention map sparse
GRADE_NAMES   = ['0 No DR', '1 Mild', '2 Moderate', '3 Severe', '4 PDR']

## 2. Preprocessing (fundus crop → square → resize → Ben Graham or CLAHE)

In [ ]:
def preprocess_fundus(img_path, size=IMG_SIZE, mode=PREPROC):
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f'Cannot read {img_path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if max(h, w) > 1600:                                   # speed: shrink very large photos first
        s = 1600 / max(h, w)
        img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    ys, xs = np.where(gray > 10)                           # crop black border
    if len(xs):
        img = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = img.shape[:2]; s = max(h, w)
    sq = np.zeros((s, s, 3), np.uint8)
    sq[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = img
    img = cv2.resize(sq, (size, size), interpolation=cv2.INTER_AREA)

    if mode == 'ben':
        img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), size / 30), -4, 128)
        circle = np.zeros((size, size), np.uint8)
        cv2.circle(circle, (size // 2, size // 2), int(size * 0.48), 1, -1)
        img = img * circle[..., None] + 128 * (1 - circle[..., None])
        img = img.astype(np.uint8)
    else:
        r, g, b = cv2.split(img)
        g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
        img = cv2.bilateralFilter(cv2.merge((r, g, b)), d=9, sigmaColor=75, sigmaSpace=75)
    return img

def build_cache(paths, name):
    cache = f'{name}_{PREPROC}_{IMG_SIZE}_{len(paths)}.npy'
    if os.path.exists(cache):
        return np.load(cache)
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
        arr = np.stack(list(tqdm(ex.map(preprocess_fundus, paths), total=len(paths), desc=f'Preprocessing {name}')))
    np.save(cache, arr)
    return arr

## 3. Load datasets (IDRiD for the main task, APTOS-2019 for pretraining)

In [ ]:
def index_images(root):
    paths = []
    for ext in ('jpg', 'jpeg', 'png', 'tif', 'tiff'):
        for e in (ext, ext.upper()):
            paths += glob.glob(os.path.join(root, '**', f'*.{e}'), recursive=True)
    return sorted(set(paths))

def signature(stem):
    nums = ''.join(re.findall(r'\d+', stem))
    return f'{nums}_test' if 'test' in stem else nums

def load_labelled(root, id_col='id_code', grade_col='diagnosis'):
    """Find every CSV with id/grade columns under root and match rows to image files."""
    paths = index_images(root)
    stem_to_path = {os.path.splitext(os.path.basename(p))[0].strip().lower(): p for p in paths}
    sig_to_path = {signature(s): p for s, p in stem_to_path.items()}
    frames = []
    for c in glob.glob(os.path.join(root, '**', '*.csv'), recursive=True):
        cols = pd.read_csv(c, nrows=1).columns
        if id_col in cols and grade_col in cols:
            frames.append(pd.read_csv(c)[[id_col, grade_col]])
    if not frames:
        raise RuntimeError(f'No CSV with columns {id_col}/{grade_col} under {root}')
    lab = pd.concat(frames).dropna().drop_duplicates(id_col)
    def match(raw):
        stem = os.path.splitext(str(raw).strip().lower())[0]
        return stem_to_path.get(stem) or sig_to_path.get(signature(stem))
    lab['img_path'] = lab[id_col].map(match)
    n_missing = lab['img_path'].isna().sum()
    lab = lab.dropna(subset=['img_path']).drop_duplicates('img_path').reset_index(drop=True)
    lab['label'] = lab[grade_col].astype(int)
    print(f'{root}: {len(paths)} images, {len(lab)} labelled matches ({n_missing} rows unmatched)')
    return lab

idrid_df = load_labelled(kagglehub.dataset_download('mariaherrerot/idrid-dataset'))
idrid_imgs = build_cache(list(idrid_df['img_path']), 'idrid')
idrid_y = idrid_df['label'].values

if USE_APTOS:
    aptos_df = load_labelled(kagglehub.dataset_download('mariaherrerot/aptos2019'))
    aptos_imgs = build_cache(list(aptos_df['img_path']), 'aptos')
    aptos_y = aptos_df['label'].values

fig, ax = plt.subplots(1, 2 if USE_APTOS else 1, figsize=(13, 3.5), squeeze=False)
for a, (name, yy) in zip(ax[0], [('IDRiD', idrid_y)] + ([('APTOS-2019', aptos_y)] if USE_APTOS else [])):
    cnt = np.bincount(yy, minlength=NUM_CLASSES)
    sns.barplot(x=GRADE_NAMES, y=cnt, color='steelblue', ax=a); a.set_title(f'{name} grade distribution (n={len(yy)})')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for a, j in zip(axes, np.random.choice(len(idrid_df), 4, replace=False)):
    a.imshow(idrid_imgs[j]); a.set_title(f'IDRiD grade {idrid_y[j]} ({PREPROC})'); a.axis('off')
plt.tight_layout(); plt.show()

## 4. Held-out test split, augmentations and loaders
15% of IDRiD is locked away as the **test set** and is never used for training, early stopping or threshold fitting.

In [ ]:
all_idx = np.arange(len(idrid_df))
trainval_idx, test_idx = train_test_split(all_idx, test_size=0.15, stratify=idrid_y, random_state=SEED)
print(f'IDRiD train+val: {len(trainval_idx)} | test (held out): {len(test_idx)}')

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomAffine(degrees=180, translate=(0.03, 0.03), scale=(0.95, 1.10)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.08)),
])
eval_tf = T.Compose([T.ToPILImage(), T.ToTensor(), T.Normalize(MEAN, STD)])

class FundusDataset(Dataset):
    def __init__(self, imgs, labels, indices, tf):
        self.imgs, self.labels, self.indices, self.tf = imgs, labels, indices, tf
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        return self.tf(self.imgs[j]), int(self.labels[j])

def make_loader(imgs, labels, indices, train):
    return DataLoader(FundusDataset(imgs, labels, indices, train_tf if train else eval_tf),
                      batch_size=BATCH_SIZE, shuffle=train, drop_last=train,
                      num_workers=2, pin_memory=True, persistent_workers=True)

def class_weights_for(labels):
    freq = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    w = 1.0 / np.sqrt(np.maximum(freq, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=device)

## 5. Model: Cross-Expert Lesion-Aware network with dual (classification + ordinal regression) head

In [ ]:
class CrossExpertDRNet(nn.Module):
    def __init__(self, backbone=BACKBONE, num_classes=NUM_CLASSES, dim=256, heads=4, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, features_only=True)
        c = self.backbone.feature_info.channels()[-1]
        self.tap = nn.Identity()                                   # Grad-CAM target
        self.proj = nn.Sequential(nn.Conv2d(c, dim, 1), nn.BatchNorm2d(dim), nn.GELU())
        self.lesion_head = nn.Sequential(
            nn.Conv2d(dim, dim // 4, 3, padding=1), nn.GELU(), nn.Conv2d(dim // 4, 1, 1))
        self.global_expert = nn.TransformerEncoderLayer(
            dim, heads, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
        self.g2l = nn.MultiheadAttention(dim, heads, dropout=0.1, batch_first=True)
        self.l2g = nn.MultiheadAttention(dim, heads, dropout=0.1, batch_first=True)
        self.norm_g, self.norm_l = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.W_G, self.W_L = nn.Linear(dim, dim), nn.Linear(dim, dim)
        self.gate = nn.Linear(2 * dim, 1)
        self.cls_head = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.3), nn.Linear(dim, num_classes))
        self.reg_head = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.3), nn.Linear(dim, 1))

    def forward(self, x):
        f = self.proj(self.tap(self.backbone(x)[-1]))              # B, D, h, w
        B, D, h, w = f.shape
        tokens = f.flatten(2).transpose(1, 2)                      # B, N, D
        m = torch.sigmoid(self.lesion_head(f))                     # lesion-attention map
        wts = m.flatten(2).transpose(1, 2)
        G = self.global_expert(tokens)                             # global retinal context expert
        L = tokens * wts                                           # lesion-aware expert
        Zg, _ = self.g2l(G, L, L)                                  # global <- lesion
        Zl, _ = self.l2g(L, G, G)                                  # lesion <- global
        G, L = self.norm_g(G + Zg), self.norm_l(L + Zl)
        v_g = G.mean(1)
        v_l = (L * wts).sum(1) / (wts.sum(1) + 1e-6)
        g = torch.sigmoid(self.gate(torch.cat([v_g, v_l], -1)))    # adaptive MoE gate
        fused = F.gelu(g * self.W_G(v_g) + (1 - g) * self.W_L(v_l))
        logits = self.cls_head(fused)
        reg = self.reg_head(fused).squeeze(-1) + 2.0               # centred on middle grade
        mask = F.interpolate(m, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return logits, reg, mask, g

_m = CrossExpertDRNet().to(device)
with torch.no_grad():
    _out = _m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device))
print('logits', tuple(_out[0].shape), '| reg', tuple(_out[1].shape), '| mask', tuple(_out[2].shape))
print(f'Parameters: {sum(p.numel() for p in _m.parameters()) / 1e6:.1f} M')
del _m, _out; torch.cuda.empty_cache()

## 6. Training utilities: dual loss, EMA, TTA prediction, threshold optimisation

In [ ]:
class EMA:
    """Exponential moving average of model weights."""
    def __init__(self, model, decay):
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.decay = decay
    @torch.no_grad()
    def update(self, model):
        src = model.state_dict()
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(src[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(src[k])

@torch.no_grad()
def predict(model, imgs, indices, tta=True):
    """Returns (class probabilities, regression score) averaged over TTA flips."""
    model.eval()
    P, R = [], []
    for s in range(0, len(indices), BATCH_SIZE):
        xb = torch.stack([eval_tf(imgs[j]) for j in indices[s:s + BATCH_SIZE]]).to(device)
        views = [xb, xb.flip(-1), xb.flip(-2), xb.flip(-1, -2)] if tta else [xb]
        p = r = 0
        for v in views:
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
                lo, rg, _, _ = model(v)
            p = p + lo.float().softmax(-1); r = r + rg.float()
        P.append((p / len(views)).cpu()); R.append((r / len(views)).cpu())
    return torch.cat(P).numpy(), torch.cat(R).numpy()

def grade_score(probs, reg):
    """Continuous grade: average of expected class and regression output."""
    return 0.5 * (probs @ np.arange(NUM_CLASSES)) + 0.5 * reg

def apply_thresholds(score, thr):
    return np.digitize(score, thr)

def fit_thresholds(score, y_true, metric=accuracy_score, rounds=4):
    thr = np.array([0.5, 1.5, 2.5, 3.5])
    best = metric(y_true, apply_thresholds(score, thr))
    for _ in range(rounds):
        for i in range(len(thr)):
            lo = thr[i - 1] + 0.01 if i > 0 else -0.5
            hi = thr[i + 1] - 0.01 if i < len(thr) - 1 else 4.5
            for c in np.linspace(lo, hi, 80):
                t = thr.copy(); t[i] = c
                sc = metric(y_true, apply_thresholds(score, t))
                if sc > best:
                    best, thr = sc, t
    return thr, best

def train_model(model, imgs, labels, tr_idx, va_idx, epochs, lr, ema_decay, tag):
    tr_loader = make_loader(imgs, labels, tr_idx, train=True)
    bb_ids = {id(p) for p in model.backbone.parameters()}
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': lr / 3},
        {'params': [p for p in model.parameters() if id(p) not in bb_ids], 'lr': lr}], weight_decay=1e-4)
    steps = epochs * len(tr_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=[lr / 3, lr], total_steps=steps,
                                                    pct_start=0.1, anneal_strategy='cos')
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    ce = nn.CrossEntropyLoss(weight=class_weights_for(labels[tr_idx]), label_smoothing=0.05)
    ema = EMA(model, ema_decay)

    hist = {k: [] for k in ('train_loss', 'train_acc', 'val_acc', 'val_qwk')}
    best, best_state, bad = -1.0, None, 0
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time()
        tot, correct, n = 0.0, 0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
                logits, reg, mask, _ = model(xb)
                loss = (ce(logits.float(), yb)
                        + REG_WEIGHT * F.smooth_l1_loss(reg.float(), yb.float())
                        + SPARSITY * mask.float().mean())
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            ema.update(model)
            tot += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)

        vp, vr = predict(ema.model, imgs, va_idx, tta=False)
        vpred = apply_thresholds(grade_score(vp, vr), np.array([0.5, 1.5, 2.5, 3.5]))
        vacc = accuracy_score(labels[va_idx], vpred)
        vqwk = cohen_kappa_score(labels[va_idx], vpred, weights='quadratic')
        for k, v in zip(hist, (tot / n, correct / n, vacc, vqwk)):
            hist[k].append(v)
        score = vacc + vqwk                                     # select on accuracy + kappa
        flag = ''
        if score > best:
            best, best_state, bad, flag = score, copy.deepcopy(ema.model.state_dict()), 0, '  <- best'
        else:
            bad += 1
        print(f'[{tag}] ep {ep:02d}/{epochs} | loss {tot / n:.3f} train-acc {correct / n:.3f} | '
              f'val acc {vacc:.3f} QWK {vqwk:.3f} | {time.time() - t0:.0f}s{flag}')
        if bad >= PATIENCE:
            print(f'[{tag}] early stop'); break
    model.load_state_dict(best_state)
    return model, hist

## 7. Stage 1: pretrain on APTOS-2019
The model learns general DR features from 3,662 images before it sees IDRiD. Skipped if `USE_APTOS = False`.

In [ ]:
PRETRAINED_PATH = 'aptos_pretrained.pth'
if USE_APTOS and not os.path.exists(PRETRAINED_PATH):
    a_tr, a_va = train_test_split(np.arange(len(aptos_y)), test_size=0.1, stratify=aptos_y, random_state=SEED)
    seed_everything()
    base = CrossExpertDRNet().to(device)
    base, aptos_hist = train_model(base, aptos_imgs, aptos_y, a_tr, a_va,
                                   epochs=APTOS_EPOCHS, lr=3e-4, ema_decay=0.999, tag='APTOS')
    torch.save(base.state_dict(), PRETRAINED_PATH)
    del base; torch.cuda.empty_cache()
    print('Saved', PRETRAINED_PATH)
elif USE_APTOS:
    print('Using existing', PRETRAINED_PATH)

## 8. Stage 2: 5-fold cross-validation fine-tuning on IDRiD
Each fold starts from the APTOS weights. Out-of-fold (OOF) predictions are used to fit decision thresholds; the held-out test set is predicted by every fold model, with TTA.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_p = np.zeros((len(trainval_idx), NUM_CLASSES)); oof_r = np.zeros(len(trainval_idx))
test_p, test_r, fold_hists = [], [], []

for fold, (tr, va) in enumerate(skf.split(trainval_idx, idrid_y[trainval_idx])):
    seed_everything(SEED + fold)
    model = CrossExpertDRNet(pretrained=not USE_APTOS).to(device)
    if USE_APTOS:
        model.load_state_dict(torch.load(PRETRAINED_PATH, map_location=device))
    model, h = train_model(model, idrid_imgs, idrid_y, trainval_idx[tr], trainval_idx[va],
                           epochs=IDRID_EPOCHS, lr=2e-4, ema_decay=0.99, tag=f'IDRiD fold {fold + 1}')
    fold_hists.append(h)
    torch.save(model.state_dict(), f'idrid_fold{fold + 1}.pth')
    oof_p[va], oof_r[va] = predict(model, idrid_imgs, trainval_idx[va], tta=True)
    p, r = predict(model, idrid_imgs, test_idx, tta=True)
    test_p.append(p); test_r.append(r)
    fold_acc = accuracy_score(idrid_y[trainval_idx[va]], oof_p[va].argmax(1))
    print(f'>>> fold {fold + 1} OOF accuracy (argmax, TTA): {fold_acc:.4f}\n')
    del model; torch.cuda.empty_cache()

test_p = np.mean(test_p, 0); test_r = np.mean(test_r, 0)
y_oof, y_test = idrid_y[trainval_idx], idrid_y[test_idx]

## 9. Decision rule selection (on out-of-fold data only)

In [ ]:
oof_score = grade_score(oof_p, oof_r)
candidates = {
    'argmax': (lambda p, r: p.argmax(1)),
    'rounded score': (lambda p, r: apply_thresholds(grade_score(p, r), np.array([0.5, 1.5, 2.5, 3.5]))),
}
thr_opt, _ = fit_thresholds(oof_score, y_oof)
candidates['optimised thresholds'] = (lambda p, r: apply_thresholds(grade_score(p, r), thr_opt))

rule_results = {name: accuracy_score(y_oof, f(oof_p, oof_r)) for name, f in candidates.items()}
print('OOF accuracy by decision rule:')
for k, v in rule_results.items():
    print(f'  {k:22s} {v:.4f}')
best_rule = max(rule_results, key=rule_results.get)
print(f'\nSelected rule: {best_rule}  (thresholds: {np.round(thr_opt, 3)})')
decide = candidates[best_rule]

In [ ]:
ep_curves = plt.figure(figsize=(16, 4.5))
for k, key in enumerate(['train_loss', 'val_acc', 'val_qwk']):
    ax = ep_curves.add_subplot(1, 3, k + 1)
    for f, h in enumerate(fold_hists):
        ax.plot(np.arange(1, len(h[key]) + 1), h[key], label=f'fold {f + 1}')
    ax.set_title(key.replace('_', ' ')); ax.set_xlabel('epoch'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 10. Final results on the held-out IDRiD test set

In [ ]:
from sklearn.metrics import (
    balanced_accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef,
    log_loss, roc_auc_score, average_precision_score, hamming_loss, zero_one_loss,
    top_k_accuracy_score, fbeta_score, confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve)
from sklearn.preprocessing import label_binarize
LABELS = list(range(NUM_CLASSES))

def compute_metrics(y_true, y_pred, y_prob):
    y_bin = label_binarize(y_true, classes=LABELS)
    m = {'Accuracy': accuracy_score(y_true, y_pred),
         'Balanced accuracy': balanced_accuracy_score(y_true, y_pred),
         'Quadratic weighted kappa': cohen_kappa_score(y_true, y_pred, weights='quadratic'),
         "Cohen's kappa": cohen_kappa_score(y_true, y_pred),
         'MCC': matthews_corrcoef(y_true, y_pred)}
    for avg in ('micro', 'macro', 'weighted'):
        m[f'Precision ({avg})'] = precision_score(y_true, y_pred, average=avg, zero_division=0)
        m[f'Recall ({avg})'] = recall_score(y_true, y_pred, average=avg, zero_division=0)
        m[f'F1 ({avg})'] = f1_score(y_true, y_pred, average=avg, zero_division=0)
    m['F-beta 0.5 (macro)'] = fbeta_score(y_true, y_pred, beta=0.5, average='macro', zero_division=0)
    m['F-beta 2.0 (macro)'] = fbeta_score(y_true, y_pred, beta=2.0, average='macro', zero_division=0)
    m['Within-one-grade accuracy'] = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)) <= 1)
    m['Top-2 accuracy'] = top_k_accuracy_score(y_true, y_prob, k=2, labels=LABELS)
    m['Hamming loss'] = hamming_loss(y_true, y_pred)
    m['Zero-one loss'] = zero_one_loss(y_true, y_pred)
    m['Log loss'] = log_loss(y_true, np.clip(y_prob, 1e-7, 1), labels=LABELS)
    present = y_bin.sum(0) > 0
    try:
        m['ROC AUC (macro, OvR)'] = roc_auc_score(y_bin[:, present], y_prob[:, present], average='macro')
        m['ROC AUC (weighted, OvR)'] = roc_auc_score(y_bin[:, present], y_prob[:, present], average='weighted')
        m['Average precision (macro)'] = average_precision_score(y_bin[:, present], y_prob[:, present], average='macro')
        m['Average precision (micro)'] = average_precision_score(y_bin, y_prob, average='micro')
    except ValueError as e:
        print('AUC metrics skipped:', e)
    return m

pred_test = decide(test_p, test_r)
m5 = compute_metrics(y_test, pred_test, test_p)
metrics_df = pd.DataFrame(list(m5.items()), columns=['Metric', 'Value']).round(4)
print(f'=== 5-class DR grading | held-out test set: {len(y_test)} images | rule: {best_rule} ===')
display(metrics_df)
print(classification_report(y_test, pred_test, labels=LABELS, target_names=GRADE_NAMES, zero_division=0))

### Referable DR (grade ≥ 2) — screening task
`P(referable) = P(grade 2) + P(grade 3) + P(grade 4)`. The threshold is chosen on out-of-fold data (Youden's J), then applied to the test set.

In [ ]:
ref_oof = oof_p[:, 2:].sum(1); ref_test = test_p[:, 2:].sum(1)
yb_oof, yb_test = (y_oof >= 2).astype(int), (y_test >= 2).astype(int)
fpr_o, tpr_o, thr_o = roc_curve(yb_oof, ref_oof)
ref_thr = thr_o[np.argmax(tpr_o - fpr_o)]
pb_test = (ref_test >= ref_thr).astype(int)
tn, fp, fn, tp = confusion_matrix(yb_test, pb_test, labels=[0, 1]).ravel()
m2 = {'Accuracy': (tp + tn) / len(yb_test),
      'Sensitivity (recall)': tp / max(tp + fn, 1),
      'Specificity': tn / max(tn + fp, 1),
      'Precision (PPV)': tp / max(tp + fp, 1),
      'NPV': tn / max(tn + fn, 1),
      'F1': f1_score(yb_test, pb_test),
      'ROC AUC': roc_auc_score(yb_test, ref_test),
      'Threshold (from OOF)': ref_thr}
print('=== Referable DR (grade >= 2 vs < 2) | held-out test set ===')
display(pd.DataFrame(list(m2.items()), columns=['Metric', 'Value']).round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(21, 6))
cm = confusion_matrix(y_test, pred_test, labels=LABELS)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0], xticklabels=GRADE_NAMES, yticklabels=GRADE_NAMES)
ax[0].set_title('5-class confusion matrix (test)'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')
sns.heatmap(np.array([[tn, fp], [fn, tp]]), annot=True, fmt='d', cmap='Greens', ax=ax[1],
            xticklabels=['Non-referable', 'Referable'], yticklabels=['Non-referable', 'Referable'])
ax[1].set_title('Referable DR confusion matrix (test)'); ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True')
y_bin = label_binarize(y_test, classes=LABELS)
colors = plt.cm.tab10(np.arange(NUM_CLASSES))
for i in LABELS:
    if y_bin[:, i].sum():
        f_, t_, _ = roc_curve(y_bin[:, i], test_p[:, i])
        ax[2].plot(f_, t_, color=colors[i], label=f'{GRADE_NAMES[i]} (AUC {auc(f_, t_):.2f})')
f_, t_, _ = roc_curve(yb_test, ref_test)
ax[2].plot(f_, t_, 'k-', lw=3, label=f'Referable DR (AUC {auc(f_, t_):.2f})')
ax[2].plot([0, 1], [0, 1], 'k:'); ax[2].set_title('ROC curves (test)')
ax[2].set_xlabel('False positive rate'); ax[2].set_ylabel('True positive rate'); ax[2].legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
print('================ SUMMARY (held-out IDRiD test set) ================')
print(f"5-class grading   accuracy {m5['Accuracy']*100:6.2f}% | QWK {m5['Quadratic weighted kappa']:.3f} "
      f"| within-one-grade {m5['Within-one-grade accuracy']*100:.1f}%")
print(f"Referable DR      accuracy {m2['Accuracy']*100:6.2f}% | sensitivity {m2['Sensitivity (recall)']*100:.1f}% "
      f"| specificity {m2['Specificity']*100:.1f}% | AUC {m2['ROC AUC']:.3f}")
print(f'Test images: {len(y_test)}. Ensemble of {N_FOLDS} folds x 4 TTA views. APTOS pretraining: {USE_APTOS}.')

## 11. Explainability: Grad-CAM + lesion-attention (fold-1 model)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

class LogitsOnly(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return self.m(x)[0]

cam_model = CrossExpertDRNet(pretrained=False).to(device)
cam_model.load_state_dict(torch.load('idrid_fold1.pth', map_location=device)); cam_model.eval()
cam = GradCAM(model=LogitsOnly(cam_model), target_layers=[cam_model.tap])

show = np.random.RandomState(0).choice(len(test_idx), min(6, len(test_idx)), replace=False)
fig, axes = plt.subplots(len(show), 3, figsize=(13, 4.2 * len(show)))
for r_, k in enumerate(show):
    j = test_idx[k]
    x = eval_tf(idrid_imgs[j]).unsqueeze(0).to(device)
    with torch.no_grad():
        _, _, mask, g = cam_model(x)
    heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(int(pred_test[k]))])[0]
    orig = cv2.resize(cv2.cvtColor(cv2.imread(idrid_df.img_path[j]), cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE))
    axes[r_, 0].imshow(orig)
    axes[r_, 0].set_title(f'True {GRADE_NAMES[y_test[k]]} | Ensemble pred {GRADE_NAMES[pred_test[k]]}')
    axes[r_, 1].imshow(show_cam_on_image(idrid_imgs[j].astype(np.float32) / 255, heat, use_rgb=True))
    axes[r_, 1].set_title('Grad-CAM')
    axes[r_, 2].imshow(orig); axes[r_, 2].imshow(mask[0, 0].cpu().numpy(), cmap='jet', alpha=0.45)
    axes[r_, 2].set_title(f'Lesion-attention map (gate g={g.item():.2f})')
    for a in axes[r_]: a.axis('off')
plt.tight_layout(); plt.show()

## 12. Per-image predictions (test set) and saving
Research output only, **not a clinical diagnosis**.

In [ ]:
summary_df = pd.DataFrame({
    'Image': [os.path.basename(idrid_df.img_path[j]) for j in test_idx],
    'True grade': [GRADE_NAMES[v] for v in y_test],
    'Predicted grade': [GRADE_NAMES[v] for v in pred_test],
    'Correct': pred_test == y_test,
    'P(referable)': ref_test.round(3),
    'Referable pred': np.where(pb_test == 1, 'Refer', 'No refer'),
    'Continuous grade score': grade_score(test_p, test_r).round(2),
})
display(summary_df)
summary_df.to_csv('test_predictions_v2.csv', index=False)
metrics_df.to_csv('test_metrics_5class_v2.csv', index=False)
print('Saved test_predictions_v2.csv, test_metrics_5class_v2.csv and idrid_fold*.pth')

# Optional: keep results after the Colab session ends
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/NITSCI
# !cp idrid_fold*.pth aptos_pretrained.pth test_*_v2.csv /content/drive/MyDrive/NITSCI/